In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load dataset

In [ ]:
df = pd.read_csv("../data_raw/transit_2026_raw.csv")

In [ ]:
# standardize columns 
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
df.columns

## Missingness inspection

In [ ]:
df.isna().mean() * 100

In [ ]:
cols = ["building", "floor", "room", "section"]

for c in cols:
    sl = f"sl_{c}_local_name"
    el = f"el_{c}_local_name"
    
    same_missing = (df[sl].isna() == df[el].isna()).mean() * 100
    print(f"{c}: {same_missing:.2f}% ens missingness")

In [ ]:
pd.set_option("display.max_rows", None)

unique_rooms = sorted(df["sl_room_local_name"].dropna().unique().tolist())
unique_rooms

In [ ]:
def categorize_room(x):
    x = str(x).lower()
    
    if "corridor" in x:
        return "corridor"
    elif "depot" in x:
        return "depot"
    elif "elevator" in x:
        return "elevator area"
    elif "storage" in x:
        return "storage"
    elif "repair" in x:
        return "repair"
    elif "waiting" in x:
        return "waiting area"
    else:
        return "other"

df["room_category"] = df["sl_room_local_name"].apply(categorize_room)

df["room_category"].value_counts()

In [ ]:
others = df[df["room_category"] == "other"]["sl_room_local_name"].dropna().unique()

for o in sorted(others):
    print(o)

In [ ]:
def categorize_room(x):
    x = str(x).lower()
    
    if "corridor" in x:
        return "corridor"
    elif "depot" in x:
        return "depot"
    elif "elevator" in x or "lift" in x:
        return "elevator/service"
    elif "storage" in x:
        return "storage"
    elif "repair" in x or "mtkf" in x:
        return "repair/technical"
    elif "waiting" in x:
        return "waiting area"
    
    # NYE grupper 👇
    elif "ok" in x:
        return "operating room"
    elif "angiography" in x:
        return "treatment room"
    elif "recovery" in x:
        return "recovery"
    elif "nursing" in x:
        return "nursing station"
    elif "waste" in x:
        return "service room"
    
    # strukturerede room IDs (meget vigtigt)
    elif any(char.isdigit() for char in x):
        return "patient/clinical room"
    
    else:
        return "other"

In [ ]:
df["start_high_res"] = df["sl_room_local_name"].notna().astype(int)
df["end_high_res"]   = df["el_room_local_name"].notna().astype(int)

In [ ]:
df["enter_high_res"] = ((df["start_high_res"] == 0) & (df["end_high_res"] == 1)).astype(int)

df["leave_high_res"] = ((df["start_high_res"] == 1) & (df["end_high_res"] == 0)).astype(int)

df["stay_high_res"]  = ((df["start_high_res"] == 1) & (df["end_high_res"] == 1)).astype(int)

df["stay_low_res"]   = ((df["start_high_res"] == 0) & (df["end_high_res"] == 0)).astype(int)

In [ ]:
df[["enter_high_res", "leave_high_res", "stay_high_res", "stay_low_res"]].mean() * 100

In [ ]:
df["end_room_category"] = df["el_room_local_name"].apply(categorize_room)
df["end_room_category"] = df["end_room_category"].fillna("no_room_info")